#### Advanced Aggregation & Pivot Tables

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

tips = sns.load_dataset("tips")

tips.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [3]:
tips.columns

Index(['total_bill', 'tip', 'sex', 'smoker', 'day', 'time', 'size'], dtype='object')

In [4]:
tips.groupby("day")["total_bill"].agg(
    ["mean", "sum", "min", "max"])

,mean,sum,min,max
day,,,,
Thur,17.682742,1096.33,7.51,43.11
Fri,17.151579,325.88,5.75,40.17
Sat,20.441379,1778.40,3.07,50.81
Sun,21.410000,1627.16,7.25,48.17


In [5]:
day_report = tips.groupby("day").agg(
    avg_bill=("total_bill", "mean"),
    total_sales=("total_bill", "sum"),
    max_bill=("total_bill", "max"),
    min_bill=("total_bill", "min"),
    avg_tip=("tip", "mean"),
    transactions=("total_bill", "count")
)

day_report

,avg_bill,total_sales,max_bill,min_bill,avg_tip,transactions
day,,,,,,
Thur,17.682742,1096.33,43.11,7.51,2.771452,62
Fri,17.151579,325.88,40.17,5.75,2.734737,19
Sat,20.441379,1778.40,50.81,3.07,2.993103,87
Sun,21.410000,1627.16,48.17,7.25,3.255132,76


In [6]:
# Multiple Dimensions पर Report
# अब थोड़ा advanced analysis: Day और smoker status के combination में performance कैसी है?

In [7]:
day_smoker_report = tips.groupby(["day", "smoker"]).agg(
    avg_bill=("total_bill", "mean"),
    total_sales=("total_bill", "sum"),
    avg_tip=("tip", "mean"),
    transactions=("total_bill", "count"))

day_smoker_report

avg_bill  total_sales   avg_tip  transactions
day  smoker                                                
Thur Yes     19.190588       326.24  3.030000            17
     No      17.113111       770.09  2.673778            45
Fri  Yes     16.813333       252.20  2.714000            15
     No      18.420000        73.68  2.812500             4
Sat  Yes     21.276667       893.62  2.875476            42
     No      19.661778       884.78  3.102889            45
Sun  Yes     24.120000       458.28  3.516842            19
     No      20.506667      1168.88  3.167895            57

In [8]:
day_smoker_report = (
    tips.groupby(["day", "smoker"]).agg(
            avg_bill=("total_bill", "mean"),
            total_sales=("total_bill", "sum"),
            avg_tip=("tip", "mean"),
            transactions=("total_bill", "count")).reset_index())

day_smoker_report

,day,smoker,avg_bill,total_sales,avg_tip,transactions
0,Thur,Yes,19.190588,326.24,3.030000,17
1,Thur,No,17.113111,770.09,2.673778,45
2,Fri,Yes,16.813333,252.20,2.714000,15
3,Fri,No,18.420000,73.68,2.812500,4
4,Sat,Yes,21.276667,893.62,2.875476,42
5,Sat,No,19.661778,884.78,3.102889,45
6,Sun,Yes,24.120000,458.28,3.516842,19
7,Sun,No,20.506667,1168.88,3.167895,57


In [9]:
# सबसे Basic Pivot Table
# प्रत्येक day पर male और female customers का average bill क्या है?
pd.pivot_table(
    tips,
    values="total_bill",
    index="day",
    columns="sex",
    aggfunc="mean")

sex,Male,Female
day,,
Thur,18.714667,16.715312
Fri,19.857000,14.145556
Sat,20.802542,19.680357
Sun,21.887241,19.872222


In [10]:
# Average Tip का Pivot
pd.pivot_table(
    tips,
    values="tip",
    index="day",
    columns="sex",
    aggfunc="mean"
)

sex,Male,Female
day,,
Thur,2.980333,2.575625
Fri,2.693000,2.781111
Sat,3.083898,2.801786
Sun,3.220345,3.367222


In [11]:
# अब average नहीं, total sales चाहिए:
pd.pivot_table(
    tips,
    values="total_bill",
    index="day",
    columns="sex",
    aggfunc="sum"
)

sex,Male,Female
day,,
Thur,561.44,534.89
Fri,198.57,127.31
Sat,1227.35,551.05
Sun,1269.46,357.70


In [12]:
# हर day पर Male/Female कितनी transactions हुईं?
pd.pivot_table(
    tips,
    values="total_bill",
    index="day",
    columns="sex",
    aggfunc="count"
)

sex,Male,Female
day,,
Thur,30,32
Fri,10,9
Sat,59,28
Sun,58,18


In [13]:
# aggfunc में Multiple Functions,अब एक ही pivot में: Average, Sum, Maximum
pd.pivot_table(
    tips,
    values="total_bill",
    index="day",
    columns="sex",
    aggfunc=["mean", "sum", "max"]
)

mean                 sum            max       
sex        Male     Female     Male  Female   Male Female
day                                                      
Thur  18.714667  16.715312   561.44  534.89  41.19  43.11
Fri   19.857000  14.145556   198.57  127.31  40.17  22.75
Sat   20.802542  19.680357  1227.35  551.05  50.81  44.30
Sun   21.887241  19.872222  1269.46  357.70  48.17  35.26

In [14]:
# Multiple Values, हम एक साथ total_bill और tip analyze कर सकते हैं:
pd.pivot_table(
    tips,
    values=["total_bill", "tip"],
    index="day",
    columns="sex",
    aggfunc="mean"
)

tip           total_bill           
sex       Male    Female       Male     Female
day                                           
Thur  2.980333  2.575625  18.714667  16.715312
Fri   2.693000  2.781111  19.857000  14.145556
Sat   3.083898  2.801786  20.802542  19.680357
Sun   3.220345  3.367222  21.887241  19.872222

In [15]:
# Day + Time के आधार पर Male/Female average bill क्या है?
pd.pivot_table(
    tips,
    values="total_bill",
    index=["day", "time"],
    columns="sex",
    aggfunc="mean"
)

sex               Male     Female
day  time                        
Thur Lunch   18.714667  16.648710
     Dinner        NaN  18.780000
Fri  Lunch   11.386667  13.940000
     Dinner  23.487143  14.310000
Sat  Dinner  20.802542  19.680357
Sun  Dinner  21.887241  19.872222

In [16]:
# fill_value
# कभी-कभी pivot table में missing combinations हो सकते हैं।

pd.pivot_table(
    tips,
    values="total_bill",
    index="day",
    columns="time",
    aggfunc="mean",
    fill_value=0 #जहाँ value available नहीं होगी, वहाँ 0
)

time,Lunch,Dinner
day,,
Thur,17.664754,18.780000
Fri,12.845714,19.663333
Sat,0.000000,20.441379
Sun,0.000000,21.410000


In [17]:
# Grand Total — margins=True
# Excel Pivot Table में अक्सर Grand Total होता है।
pd.pivot_table(
    tips,
    values="total_bill",
    index="day",
    columns="sex",
    aggfunc="mean",
    margins=True
)

# margins=True overall total/summary भी add करता है।

sex,Male,Female,All
day,,,
Thur,18.714667,16.715312,17.682742
Fri,19.857000,14.145556,17.151579
Sat,20.802542,19.680357,20.441379
Sun,21.887241,19.872222,21.410000
All,20.744076,18.056897,19.785943


In [18]:
# Pivot Table को Sort करना
# मान लो हमें day-wise average bill पता करना है।
pivot = pd.pivot_table(
    tips,
    values="total_bill",
    index="day",
    aggfunc="mean")

pivot

,total_bill
day,
Thur,17.682742
Fri,17.151579
Sat,20.441379
Sun,21.410000


In [19]:
pivot.sort_values(
    "total_bill",
    ascending=False
)

,total_bill
day,
Sun,21.410000
Sat,20.441379
Thur,17.682742
Fri,17.151579


In [20]:
# groupby() vs pivot_table()

In [21]:
tips.groupby("day")["total_bill"].mean()

day
Thur    17.682742
Fri     17.151579
Sat     20.441379
Sun     21.410000
Name: total_bill, dtype: float64

In [22]:
pd.pivot_table(
    tips,
    values="total_bill",
    index="day",
    columns="sex",
    aggfunc="mean"
)

sex,Male,Female
day,,
Thur,18.714667,16.715312
Fri,19.857000,14.145556
Sat,20.802542,19.680357
Sun,21.887241,19.872222


In [23]:
# मुझे बताओ कि अलग-अलग days पर Male और Female customers का average spending क्या है."

tips.groupby(["day", "sex"])["total_bill"].mean()

day   sex   
Thur  Male      18.714667
      Female    16.715312
Fri   Male      19.857000
      Female    14.145556
Sat   Male      20.802542
      Female    19.680357
Sun   Male      21.887241
      Female    19.872222
Name: total_bill, dtype: float64

In [24]:
pd.pivot_table(
    tips,
    values="total_bill",
    index="day",
    columns="sex",
    aggfunc="mean"
)

sex,Male,Female
day,,
Thur,18.714667,16.715312
Fri,19.857000,14.145556
Sat,20.802542,19.680357
Sun,21.887241,19.872222


In [25]:
# एक और Business Question
# किस day और time पर average bill सबसे ज्यादा है?"

pd.pivot_table(
    tips,
    values="total_bill",
    index="day",
    columns="time",
    aggfunc="mean"
)

time,Lunch,Dinner
day,,
Thur,17.664754,18.780000
Fri,12.845714,19.663333
Sat,NaN,20.441379
Sun,NaN,21.410000


In [26]:
2*5

10

In [27]:
# Transaction Count Pivot
transaction_pivot = pd.pivot_table(
    tips,
    values="total_bill",
    index="day",
    columns="time",
    aggfunc="count",
    fill_value=0
)

transaction_pivot

time,Lunch,Dinner
day,,
Thur,61,1
Fri,7,12
Sat,0,87
Sun,0,76


In [28]:
# Complete Management Dashboard Table
management_report = tips.groupby("day").agg(
    avg_bill=("total_bill", "mean"),
    total_sales=("total_bill", "sum"),
    avg_tip=("tip", "mean"),
    total_tip=("tip", "sum"),
    transactions=("total_bill", "count"),
    avg_party_size=("size", "mean")).reset_index()

management_report

,day,avg_bill,total_sales,avg_tip,total_tip,transactions,avg_party_size
0,Thur,17.682742,1096.33,2.771452,171.83,62,2.451613
1,Fri,17.151579,325.88,2.734737,51.96,19,2.105263
2,Sat,20.441379,1778.40,2.993103,260.40,87,2.517241
3,Sun,21.410000,1627.16,3.255132,247.39,76,2.842105


In [29]:
management_report.sort_values("total_sales",ascending=False)

,day,avg_bill,total_sales,avg_tip,total_tip,transactions,avg_party_size
2,Sat,20.441379,1778.40,2.993103,260.40,87,2.517241
3,Sun,21.410000,1627.16,3.255132,247.39,76,2.842105
0,Thur,17.682742,1096.33,2.771452,171.83,62,2.451613
1,Fri,17.151579,325.88,2.734737,51.96,19,2.105263


Data Analysis में अक्सर workflow ऐसा होता है:
Raw Data
   ↓
Filter
   ↓
Groupby / Pivot
   ↓
Aggregation
   ↓
Sort
   ↓
Visualization
   ↓
Business Insight

In [30]:
filtered = tips[tips["total_bill"] > 20]

report = pd.pivot_table(
    filtered,
    values="total_bill",
    index="day",
    columns="sex",
    aggfunc="mean")

report

sex,Male,Female
day,,
Thur,27.579,29.856667
Fri,27.984,22.750000
Sat,29.260,26.970000
Sun,28.714,28.017143


In [31]:
# cheat sheet and revision

Groupby
>df.groupby("category")["sales"].mean()
Multiple aggregation
df.groupby("category")["sales"].agg(
    ["mean", "sum", "min", "max"]
)


Named aggregation
>df.groupby("category").agg(
    avg_sales=("sales", "mean"),
    total_sales=("sales","sum"))

Pivot
>pd.pivot_table(
    df,
    values="sales",
    index="category",
    columns="region",
    aggfunc="mean")

Pivot with multiple functions
>pd.pivot_table(
    df,
    values="sales",
    index="category",
    columns="region",
    aggfunc=["mean", "sum"])

Pivot with missing values
>pd.pivot_table(
    df,
    values="sales",
    index="category",
    columns="region",
    aggfunc="mean",
    fill_value=0)

Grand Total
>pd.pivot_table(
    df,
    values="sales",
    index="category",
    columns="region",
    aggfunc="sum",
    margins=True)

#### Interview Questions
Q1. What is a Pivot Table?
>Pivot Table is a summarized table used to analyze data across multiple dimensions using aggregation functions.

Q2. What is the difference between groupby() and pivot_table()?
>groupby() primarily performs grouped aggregation.
pivot_table() creates a spreadsheet-like cross-tabulation where dimensions can be represented as rows and columns.

Q3. What does aggfunc="mean" mean?
>It tells Pandas to calculate the average for each group/cell.

Q4. What does fill_value=0 do?
>It replaces missing values in the pivot table with 0.

Q5. What does margins=True do?
>It adds overall summary values, similar to Grand Total in Excel Pivot Tables.